In [73]:
import geopandas as gpd
import folium
from folium import plugins
import pandas as pd
from pathlib import Path
import glob

# Path to the shapefiles directory
shapefile_dir = r"C:\Users\kylec\Downloads\LB_MSOA2021_shp\msoa2021"

# Get all shapefile paths
shapefiles = sorted(glob.glob(f"{shapefile_dir}/*.shp"))
print(f"Found {len(shapefiles)} borough shapefiles")

# Load all shapefiles
gdfs = []
for shp in shapefiles:
    borough = Path(shp).stem
    print(f"Loading {borough}...")
    gdf = gpd.read_file(shp)
    gdfs.append(gdf)

# Combine all borough GeoDataFrames
london_msoa = pd.concat(gdfs, ignore_index=True)
print(f"\nCombined dataset: {len(london_msoa)} MSOA polygons")
print(f"Columns: {london_msoa.columns.tolist()}")

Found 33 borough shapefiles
Loading Barking and Dagenham...
Loading Barnet...
Loading Bexley...
Loading Brent...
Loading Bromley...
Loading Camden...
Loading City of London...
Loading Croydon...
Loading Ealing...
Loading Enfield...
Loading Greenwich...
Loading Hackney...
Loading Hammersmith and Fulham...
Loading Haringey...
Loading Harrow...
Loading Havering...
Loading Hillingdon...
Loading Hounslow...
Loading Islington...
Loading Kensington and Chelsea...
Loading Kingston upon Thames...
Loading Lambeth...
Loading Lewisham...
Loading Merton...
Loading Newham...
Loading Redbridge...
Loading Richmond upon Thames...
Loading Southwark...
Loading Sutton...
Loading Tower Hamlets...
Loading Waltham Forest...
Loading Wandsworth...
Loading Westminster...

Combined dataset: 1002 MSOA polygons
Columns: ['msoa21cd', 'msoa21nm', 'lad22cd', 'lad22nm', 'geometry']


In [74]:
# Check and reproject CRS to WGS84 for folium
print(f"Current CRS: {london_msoa.crs}")

# Reproject to WGS84 (EPSG:4326) for folium compatibility
if london_msoa.crs != 'EPSG:4326':
    london_msoa = london_msoa.to_crs('EPSG:4326')
    print(f"Reprojected to: {london_msoa.crs}")
else:
    print("Already in WGS84")

Current CRS: EPSG:27700
Reprojected to: EPSG:4326


In [75]:
INNER_LONDON_BOROUGHS = [
    "City of London",
    "Camden",
    "Greenwich",
    "Hackney",
    "Hammersmith and Fulham",
    "Islington",
    "Kensington and Chelsea",
    "Lambeth",
    "Lewisham",
    "Southwark",
    "Tower Hamlets",
    "Wandsworth",
    "Westminster",
    "Newham",
]
london_msoa = london_msoa[london_msoa['lad22nm'].isin(INNER_LONDON_BOROUGHS)]

In [76]:
london_msoa

,msoa21cd,msoa21nm,lad22cd,lad22nm,geometry
166,E02000171,Camden 006,E09000007,Camden,"POLYGON ((-0.15161 51.55397, -0.15164 51.55396..."
167,E02000172,Camden 007,E09000007,Camden,"POLYGON ((-0.14854 51.55357, -0.14843 51.55346..."
168,E02000173,Camden 008,E09000007,Camden,"POLYGON ((-0.17072 51.55461, -0.17048 51.55433..."
169,E02000174,Camden 009,E09000007,Camden,"POLYGON ((-0.13546 51.5547, -0.13543 51.55468,..."
170,E02000175,Camden 010,E09000007,Camden,"POLYGON ((-0.18544 51.55193, -0.18533 51.55186..."
...,...,...,...,...,...
997,E02000982,Westminster 023,E09000033,Westminster,"POLYGON ((-0.14145 51.49476, -0.14136 51.49466..."
998,E02000983,Westminster 024,E09000033,Westminster,"MULTIPOLYGON (((-0.14922 51.48577, -0.1493 51...."
999,E02000960,Westminster 001,E09000033,Westminster,"POLYGON ((-0.17415 51.53828, -0.17415 51.53823..."
1000,E02000972,Westminster 013,E09000033,Westminster,"POLYGON ((-0.14352 51.524, -0.14352 51.52395, ..."


In [77]:
# Create borough boundaries by dissolving MSOAs
print("Creating borough boundaries from MSOAs...")
borough_boundaries = london_msoa.dissolve(by='lad22nm', as_index=False)
print(f"Created {len(borough_boundaries)} borough boundaries")

# Calculate the center of London for the map
bounds = london_msoa.total_bounds
center_lon = (bounds[0] + bounds[2]) / 2
center_lat = (bounds[1] + bounds[3]) / 2

print(f"Map center: ({center_lat}, {center_lon})")
print(f"Bounds: {bounds}")

# Create base map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles='OpenStreetMap'
)

# Add borough boundaries as outlines
for idx, row in borough_boundaries.iterrows():
    folium.GeoJson(
        data=gpd.GeoSeries([row.geometry]).__geo_interface__,
        style_function=lambda x: {
            'fillColor': 'none',
            'color': '#666666',
            'weight': 2,
            'fillOpacity': 0
        },
        popup=folium.Popup(f"<b>{row['lad22nm']}</b>", max_width=200)
    ).add_to(m)

print("Map created with borough boundaries, adding MSOA polygons...")

Creating borough boundaries from MSOAs...
Created 14 borough boundaries
Map center: (51.49439638313204, -0.06747722801493476)
Bounds: [-0.25911339 51.41100855  0.12415894 51.57778422]
Map created with borough boundaries, adding MSOA polygons...


In [78]:
# Create merged boundary of all boroughs (Inner London)
inner_london_merged = london_msoa.dissolve(as_index=False)
print(f"Created merged Inner London boundary")

# Add merged boundary as a named layer
merged_layer = folium.FeatureGroup(name='Inner London Boundary', show=True)
folium.GeoJson(
    data=gpd.GeoSeries([inner_london_merged.geometry.iloc[0]]).__geo_interface__,
    style_function=lambda x: {
        'fillColor': 'none',
        'color': '#FF0000',
        'weight': 3,
        'fillOpacity': 0,
        'dashArray': '5, 5'
    },
    popup=folium.Popup(f"<b>Inner London Boundary</b>", max_width=200)
).add_to(merged_layer)

merged_layer.add_to(m)

# Add borough boundaries as a named layer
borough_layer = folium.FeatureGroup(name='Borough Boundaries', show=True)
for idx, row in borough_boundaries.iterrows():
    folium.GeoJson(
        data=gpd.GeoSeries([row.geometry]).__geo_interface__,
        style_function=lambda x: {
            'fillColor': 'none',
            'color': '#333333',
            'weight': 2.5,
            'fillOpacity': 0
        },
        popup=folium.Popup(f"<b>{row['lad22nm']}</b>", max_width=200)
    ).add_to(borough_layer)

borough_layer.add_to(m)

# Add MSOA polygons to the map with GeoJson
msoa_layer = folium.FeatureGroup(name='MSOA Areas', show=True)
folium.GeoJson(
    data=london_msoa.to_json(),
    style_function=lambda x: {
        'fillColor': '#3388ff',
        'color': '#0033cc',
        'weight': 1,
        'fillOpacity': 0.3
    },
    highlight_function=lambda x: {
        'fillColor': '#ffaa33',
        'color': '#ff3300',
        'weight': 2,
        'fillOpacity': 0.6
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['msoa21nm', 'msoa21cd'],
        aliases=['MSOA Name', 'MSOA Code']
    )
).add_to(msoa_layer)

msoa_layer.add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

print("All layers added: Inner London Boundary, Borough Boundaries, and MSOA Areas")

Created merged Inner London boundary
All layers added: Inner London Boundary, Borough Boundaries, and MSOA Areas


In [79]:
import json
from shapely.geometry import mapping

# Get City of London borough and calculate centroid
city_of_london = london_msoa[london_msoa['lad22nm'] == 'City of London']
if len(city_of_london) > 0:
    city_merged = city_of_london.dissolve(as_index=False)
    city_centroid = city_merged.geometry.iloc[0].centroid
    centroid_coords = [city_centroid.x, city_centroid.y]
    print(f"City of London centroid: {centroid_coords}")
else:
    print("City of London not found")
    centroid_coords = None

# Get Inner London boundary geometry
inner_london_geometry = mapping(inner_london_merged.geometry.iloc[0])

# Create GeoJSON FeatureCollection with city_centroid property
inner_london_json = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {
                "name": "Inner London Boundary",
                "city_centroid": centroid_coords
            },
            "geometry": inner_london_geometry
        }
    ]
}

# Export to JSON file
output_file = "inner_london_boundary.json"
with open(output_file, 'w') as f:
    json.dump(inner_london_json, f, indent=2)

print(f"\nExported to {output_file}")
print(f"GeoJSON properties: name, city_centroid")

City of London centroid: [-0.0924108873340905, 51.514381685897675]

Exported to inner_london_boundary.json
GeoJSON properties: name, city_centroid


In [80]:
# Save the map to HTML file
output_path = "london_msoa_map.html"
m.save(output_path)
print(f"Map saved to {output_path}")

Map saved to london_msoa_map.html


In [81]:
borough_boundaries.to_csv("london_borough_boundaries.csv", index=False)
london_msoa.to_csv("london_msoa_data.csv", index=False)